# QLoRA DISTILL-ONLY + HELD-OUT — ablation (Kaggle T4)
**Mục tiêu:** kiểm 2 điều mà bản mixed (34.76 ≈ rule) chưa tách được:
1. Bỏ synthetic (nghi pha loãng) → train THUẦN 124→100 mẫu distill, 8 epoch.
2. **Đo tổng quát hoá THẬT**: train trên 80 file, chấm trên **20 file held-out (model CHƯA thấy input)** — tránh train-on-test đã thổi phồng 34.76.

Settings: **GPU T4 x2**, **Internet On**. Chạy nền: **Save Version → Save & Run All (Commit)**. ~5-6h.

In [ ]:
# 1) Code — nhánh Duckun
%cd /kaggle/working
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /kaggle/working/viettel-medreason
!git fetch -q origin Duckun && git checkout -q Duckun && git pull -q origin Duckun
!git log --oneline -1

In [ ]:
# 2) Env — giữ numpy/torch/transformers bản Kaggle; chỉ update bnb/peft/accelerate
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes peft accelerate
import numpy, torch
print('numpy',numpy.__version__,'| torch',torch.__version__,'| GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not numpy.__version__.startswith('2'):
    raise SystemExit('numpy != 2.x -> Restart & Run All')

In [ ]:
# 3) Dựng data distill-TRAIN (loại 20 file held-out) + tách gold held-out / seen
HOLDOUT='5,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100'   # 20 file, rải đều
!python src/datagen/distill_frontier.py --holdout {HOLDOUT} --out data/silver/frontier_ref/train_sft_distill_train.jsonl
import os, shutil
hold=set(HOLDOUT.split(','))
for d in ('holdout_gold','seen_gold'):
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
for n in map(str, range(1,101)):
    g=f'data/dev/gold/{n}.json'
    if os.path.exists(g):
        shutil.copy(g, f'{"holdout_gold" if n in hold else "seen_gold"}/{n}.json')
print('holdout_gold:',len(os.listdir('holdout_gold')),'| seen_gold:',len(os.listdir('seen_gold')))

In [ ]:
# 4) SMOKE (kiểm không OOM ở max_len 2560)
import os; os.environ['CUDA_VISIBLE_DEVICES']='0'
MODEL='Qwen/Qwen2.5-3B-Instruct'
!python scripts/train_qlora.py --data data/silver/frontier_ref/train_sft_distill_train.jsonl --model {MODEL} --out /kaggle/working/smoke --epochs 0.1 --max-len 2560 --seed 42

In [ ]:
# 5) TRAIN distill-only — 100 mẫu (80 file) x 8 epoch, max_len 2560. ~4-5h T4.
#    8 epoch vì mẫu ít; held-out ở cell 8 sẽ lộ nếu overfit.
ADAPTER='/kaggle/working/qwen-lora-distill'
MODEL='Qwen/Qwen2.5-3B-Instruct'
!python scripts/train_qlora.py --data data/silver/frontier_ref/train_sft_distill_train.jsonl --model {MODEL} --out {ADAPTER} --epochs 8 --batch 1 --grad-accum 8 --lr 2e-4 --max-len 2560 --seed 42
!ls -la {ADAPTER}

In [ ]:
# 6) Config -> backend llm + adapter distill
import yaml
ADAPTER='/kaggle/working/qwen-lora-distill'
cfg=yaml.safe_load(open('configs/config.yaml',encoding='utf-8'))
cfg['extract']['backend']='llm'
cfg['extract']['lora_adapter']=ADAPTER
cfg['extract']['llm_model']='Qwen/Qwen2.5-3B-Instruct'
yaml.safe_dump(cfg,open('configs/config.yaml','w',encoding='utf-8'),allow_unicode=True,sort_keys=False)
print('adapter =',cfg['extract']['lora_adapter'])

In [ ]:
# 7) Dự đoán 100 file: QLoRA (llm) + rule (mốc so). rule chạy nhanh.
!python src/pipeline.py --input data/test/input --output dev_pred  --backend llm
!python src/pipeline.py --input data/test/input --output rule_pred --backend rule

In [ ]:
# 8) GO/NO-GO — so quyet dinh la HELD-OUT (20 file model CHUA thay input)
print('===== QLoRA distill-only | HELD-OUT (20 file, tong quat hoa THAT) =====')
!python src/eval/official_scorer.py --pred dev_pred --gold holdout_gold
print(); print('===== RULE | HELD-OUT (dung 20 file do -- moc so) =====')
!python src/eval/official_scorer.py --pred rule_pred --gold holdout_gold
print(); print('===== QLoRA distill-only | SEEN (80 file da train -- muc thuoc long) =====')
!python src/eval/official_scorer.py --pred dev_pred --gold seen_gold
print()
print('>> QLoRA(held-out) > RULE(held-out) => thang THAT, dang scale.')
print('>> SEEN >> HELD-OUT nhieu => model thuoc long, can them data/model to hon.')